# 04 — Analytics: respostas às seis perguntas

Todas as respostas usam tabelas Gold e `display()`, sem bibliotecas de gráficos. Valores ausentes não são tratados como receita ou lucro zero. As consultas financeiras consideram os filmes do fato (lançados, com data realizada).


In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import re

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
def parametro(nome, padrao):
    try:
        return dbutils.widgets.get(nome)
    except Exception:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)

CATALOGO = parametro("catalogo", "workspace")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOGO):
    raise ValueError("Use um nome simples de catálogo, com letras, números e underscore.")
DATA_INICIO = datetime.strptime(parametro("data_inicio", (hoje-timedelta(days=6)).strftime("%m-%d-%Y")), "%m-%d-%Y").date()
DATA_FIM = datetime.strptime(parametro("data_fim", hoje.strftime("%m-%d-%Y")), "%m-%d-%Y").date()
if DATA_INICIO > DATA_FIM or DATA_FIM > hoje:
    raise ValueError("O período deve estar em ordem e não pode terminar no futuro.")

def tabela(camada, nome):
    return f"{CATALOGO}.{camada}.{nome}"

fato = spark.table(tabela("gold", "fact_movies_performance"))
filmes = spark.table(tabela("gold", "dim_movies"))
generos = spark.table(tabela("gold", "dim_genres"))
pessoas = spark.table(tabela("gold", "dim_people"))
produtoras = spark.table(tabela("gold", "dim_companies"))
bg = spark.table(tabela("gold", "bridge_movie_genre"))
bp = spark.table(tabela("gold", "bridge_movie_person"))
bc = spark.table(tabela("gold", "bridge_movie_company"))
filmes_lancados = filmes.filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.lit(hoje)))
limite_superior = filmes_lancados.agg(F.max("data_lancamento").alias("data_limite")).first()["data_limite"]
if limite_superior is None:
    raise RuntimeError("Não há filmes com lançamento realizado para definir os períodos.")
print("Data limite superior das análises:", limite_superior)

## 1. Receita total em reais

`sum` ignora valores NULL. Também é exibido quantos filmes têm receita válida e quantos estão sem receita para explicitar a cobertura do total.


In [ ]:
resposta_1 = fato.agg(
    F.sum("receita_brl").alias("receita_total_brl"),
    F.count("receita_brl").alias("qtd_filmes_com_receita"),
    F.sum(F.col("receita_brl").isNull().cast("int")).alias("qtd_filmes_sem_receita"))
display(resposta_1)

## 2. Cinco filmes com maior popularidade

Ordenação decrescente e `limit(5)`. O título e a chave do filme desempatarão valores iguais, garantindo cinco registros.


In [ ]:
resposta_2 = (fato.join(filmes.select("sk_movie_id", "titulo"), "sk_movie_id", "inner")
    .filter(F.col("popularidade").isNotNull())
    .orderBy(F.col("popularidade").desc(), F.col("titulo").asc(), F.col("sk_movie_id").asc())
    .select("titulo", "popularidade").limit(5))
display(resposta_2)

## 3. Quantidade de filmes por gênero

A ponte conecta gêneros aos filmes. `countDistinct` conta cada filme uma vez dentro de cada gênero. Nesta pergunta, "cada gênero possui" considera o catálogo inteiro da dimensão, incluindo filmes ainda não lançados; nas consultas financeiras apenas o fato de lançados é usado. Gêneros sem associação não aparecem na dimensão criada a partir das relações válidas.


In [ ]:
resposta_3 = (bg.join(generos, "sk_genre_id", "inner")
    .groupBy("sk_genre_id", "nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc(), "nome_genero")
    .select("nome_genero", "qtd_filmes"))
display(resposta_3)

## 4. Dez filmes com maior receita e RANK

`rank()` atribui a mesma posição a receitas iguais e pode deixar lacunas na numeração.

In [ ]:
w_rank = Window.orderBy(F.col("receita_usd").desc())
resposta_4 = (fato.filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao_ranking", F.rank().over(w_rank))
    .join(filmes.select("sk_movie_id", "titulo"), "sk_movie_id", "inner")
    .orderBy(F.col("receita_usd").desc(), F.col("titulo").asc(), F.col("sk_movie_id").asc())
    .select("posicao_ranking", "titulo", "receita_usd", "receita_brl").limit(10))
display(resposta_4)

## 5. Ator com mais participações nos últimos dois anos

Primeiro o tipo Ator é filtrado, e contamos filmes distintos no intervalo de 24 meses. O vencedor é selecionado por ranking; se houver empate no primeiro lugar, todos os líderes são exibidos, sem escolher arbitrariamente um nome.


In [ ]:
ultimos_2_anos = filmes_lancados.filter(F.col("data_lancamento").between(
    F.add_months(F.lit(limite_superior), -24), F.lit(limite_superior))).select("sk_movie_id")
participacoes = (bp.join(ultimos_2_anos, "sk_movie_id", "inner")
    .join(pessoas.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id", "inner")
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes")))
resposta_5 = (participacoes.withColumn("posicao", F.dense_rank().over(Window.orderBy(F.col("qtd_participacoes").desc())))
    .filter("posicao = 1").select("nome_pessoa", "qtd_participacoes").orderBy("nome_pessoa"))
display(resposta_5)

## 6. Produtora com maior lucro nos últimos cinco anos

O lucro dos filmes associados a cada produtora é somado, contando cada vínculo uma vez. Como a base não informa a participação de cada empresa, o **lucro integral do filme a cada produtora participante** é atribuido. Portanto, os totais das produtoras não podem ser somados para obter o lucro geral do catálogo. Essa atribuição pode ser substituída por rateio apenas se houver uma regra de negócio para isso.

Filmes sem receita ou orçamento não têm lucro conhecido e não entram na soma. Por isso, a quantidade de filmes com lucro conhecido e os sem informação são mostrados, para não confundir a soma parcial com lucro completo. Em empate, todas as líderes são exibidas.


In [ ]:
ultimos_5_anos = filmes_lancados.filter(F.col("data_lancamento").between(
    F.add_months(F.lit(limite_superior), -60), F.lit(limite_superior))).select("sk_movie_id")
lucro_produtoras = (bc.join(ultimos_5_anos, "sk_movie_id", "inner")
    .join(fato.select("sk_movie_id", "lucro_usd", "lucro_brl"), "sk_movie_id", "inner")
    .join(produtoras, "sk_company_id", "inner")
    .groupBy("sk_company_id", "nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"), F.sum("lucro_brl").alias("lucro_total_brl"),
        F.count("lucro_usd").alias("qtd_filmes_com_lucro_conhecido"),
        F.sum(F.col("lucro_usd").isNull().cast("int")).alias("qtd_filmes_sem_lucro_informado"))
    .filter(F.col("lucro_total_usd").isNotNull()))
resposta_6 = (lucro_produtoras.withColumn("posicao", F.dense_rank().over(Window.orderBy(F.col("lucro_total_usd").desc())))
    .filter("posicao = 1").select("nome_produtora", "lucro_total_usd", "lucro_total_brl",
        "qtd_filmes_com_lucro_conhecido", "qtd_filmes_sem_lucro_informado").orderBy("nome_produtora"))
display(resposta_6)